# 03 - Arquitetura Detalhada

## Descrição

Este notebook aprofunda a arquitetura técnica do projeto, analisando padrões de design, decisões arquiteturais, fluxos de execução e componentes transversais. O sistema adota uma **arquitetura medalhão (Medallion Architecture)** sobre um **Data Lake** com **object storage** (MinIO/S3), processamento via **Apache Spark** e **Delta Lake**, e orquestração pelo **Apache Airflow**.

A arquitetura é **monolítica em termos de deploy** (todos os serviços rodam em containers Docker no mesmo host), mas **modular em termos de responsabilidades** (cada camada tem função, formato e contratos distintos).

---

## Arquitetura Medalhão (Medallion Architecture)

A arquitetura medalhão é um padrão de design de Data Lake que organiza os dados em camadas com níveis crescentes de qualidade e estruturação. Cada camada representa um estágio de processamento, com responsabilidades bem definidas.

### Diagrama da Arquitetura Medalhão

```mermaid
flowchart TD
    subgraph Origem
        M["MongoDB Atlas\n10 coleções\n15K docs cada"]
    end

    subgraph Camada_Landing["Camada Landing — Cópia Fiel"]
        L["JSON Estendido\nTipos BSON preservados\nSem transformação"]
    end

    subgraph Camada_Bronze["Camada Bronze — Persistência Analítica"]
        B["Delta Lake\nMetadados de auditoria\nParticionado por ingestion_date"]
    end

    subgraph Camada_Silver["Camada Silver — Qualidade e Conformação"]
        S["Delta Lake\nLimpo, tipado, deduplicado\nIntegridade referencial\nLog de rejeições"]
    end

    subgraph Camada_Gold["Camada Gold — Modelo Analítico"]
        G["Delta Lake\nDimensões SCD Tipo 2\nFatos particionados por ano\nSurrogate keys"]
    end

    subgraph Consumo
        D["Dashboard / BI\nKPIs e Métricas"]
    end

    M -->|mongodb_to_landing| L
    L -->|landing_to_bronze| B
    B -->|bronze_to_silver| S
    S -->|silver_to_gold| G
    G -->|Consultas OLAP| D

    style M fill:#15803d,color:#fff,stroke:#166534
    style L fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style B fill:#fdba74,stroke:#b45309,color:#7c2d12
    style S fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style G fill:#fde047,stroke:#a16207,color:#713f12
    style D fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
```

### Responsabilidades de Cada Camada

#### Landing — Cópia Fiel da Origem

**O que é**: Armazenamento bruto dos dados extraídos do MongoDB, sem qualquer transformação.

**Por que existe**: Garante que a origem possa ser reprocessada a qualquer momento, preservando tipos BSON originais (ISODate, NumberInt, etc.) no formato JSON estendido.

**Como funciona**:
1. A DAG `mongodb_to_landing` conecta-se ao MongoDB via `MongoHook`
2. Aplica filtro incremental baseado no maior `updated_at` já processado (checkpoint)
3. Exporta documentos em lote para arquivo JSON Lines (MongoDB Extended JSON Canonical)
4. Grava no MinIO/S3 em prefixo particionado: `landing/ecommerce/<coleção>/extraction_date=YYYY-MM-DD/run_id=<id>/part-00000.json`
5. Atualiza o checkpoint em Airflow Variable para a próxima execução
6. Gera manifesto JSON com totais e metadados da execução

**Formato**: JSON Lines (MongoDB Extended JSON Canonical) — cada linha é um documento BSON serializado.

> **Referência**: `dags/mongodb_to_landing.py`, `dags/lib/mongodb_landing.py`

#### Bronze — Persistência Analítica com Metadados

**O que é**: Conversão dos dados da Landing para um formato analítico (Delta Lake) com colunas de auditoria.

**Por que existe**: Permitir consultas SQL-like, time-travel e operações ACID sobre os dados brutos, além de rastrear a linhagem de cada registro.

**Como funciona**:
1. A DAG `landing_to_bronze` valida a existência dos arquivos Landing e da estrutura Bronze
2. Submete job PySpark via `SparkSubmitOperator`
3. O job lê JSON estendido, extrai metadados do caminho (`extraction_date`, `run_id`)
4. Adiciona colunas de controle: `_bronze_source_file`, `_bronze_extraction_date`, `_bronze_landing_run_id`, `_bronze_airflow_run_id`, `_bronze_ingested_at`, `ingestion_date`
5. Evita reprocessamento: verifica `_bronze_source_file` já existente na tabela Bronze (left anti-join)
6. Grava em Delta Lake particionado por `ingestion_date`
7. Gera manifesto de auditoria

**Formato**: Delta Lake sobre S3A (object storage). Particionado por `ingestion_date`.

> **Referência**: `spark_jobs/landing_to_bronze.py`, `dags/landing_to_bronze.py`

#### Silver — Qualidade, Limpeza e Conformação

**O que é**: Dados limpos, tipados, deduplicados e validados, com integridade referencial e log de rejeições.

**Por que existe**: Garantir que dados inconsistentes, duplicados ou inválidos não cheguem à camada analítica, documentando todas as rejeições.

**Como funciona**:
1. A DAG `bronze_to_silver` valida a existência das tabelas Bronze e Silver
2. Submete job PySpark via `SparkSubmitOperator`
3. O job lê tabela Delta Bronze, converte tipos BSON (NumberInt, ISODate) para tipos Spark nativos
4. Aplica regras de normalização (trim, lower, upper, digits-only para CPF/CEP)
5. **Deduplicação**: mantém o registro mais recente por chave primária (`updated_at` desc, `_bronze_ingested_at` desc, `_bronze_source_file` desc)
6. **Validação de qualidade**: campos obrigatórios, domínios (enum), padrões regex, ranges numéricos
7. **Integridade referencial**: valida chaves estrangeiras contra tabelas Silver já processadas (broadcast join)
8. **Regras de unicidade de negócio**: CPF duplicado em clientes, pagamento aprovado duplicado por pedido — rejeições vão para `quality_log`
9. **MERGE incremental**: insere novos registros, atualiza alterados (compara `updated_at` + hash), ignora inalterados
10. Gera manifesto com métricas de qualidade

**Formato**: Delta Lake. Tabela `silver/_control/quality_log/` append-only para registros rejeitados.

> **Referência**: `spark_jobs/bronze_to_silver.py`, `dags/lib/bronze_silver.py` (ENTITY_RULES)

#### Gold — Modelo Dimensional Analítico

**O que é**: Modelo dimensional (Kimball) com dimensões SCD Tipo 2 e fatos, pronto para consumo em BI.

**Por que existe**: Entregar dados estruturados para análise de negócio com histórico versionado e métricas pré-calculadas.

**Como funciona**:
1. A DAG `silver_to_gold` valida a existência das tabelas Silver e Gold
2. Submete job PySpark via `SparkSubmitOperator`
3. **Dimensões**: constroí `dim_tempo`, `dim_cliente`, `dim_produto`, `dim_cupom` a partir das tabelas Silver
4. **SCD Tipo 2**: dimensões `dim_cliente`, `dim_produto`, `dim_cupom` versionam atributos — cada alteração cria nova versão, expira a anterior
5. **Fatos**: constroí `fato_vendas`, `fato_pagamentos`, `fato_entregas`, `fato_avaliacoes` com joins e métricas
6. **Surrogate keys**: fatos recebem as SKs vigentes na data do evento (point-in-time join)
7. **Particionamento**: fatos particionados por `ano` para otimização de consultas
8. **MERGE**: sincroniza dimensões e fatos sem duplicar dados em execuções subsequentes
9. Gera manifesto com métricas de sincronização

**Formato**: Delta Lake. Dimensões SCD2 com colunas de controle `dw_valid_from`, `dw_valid_to`, `dw_is_current`, `dw_record_hash`.

> **Referência**: `spark_jobs/silver_to_gold.py`, `dags/lib/silver_gold.py` (GOLD_MODELS, SCD2 constants)

---

## Padrões de Design Arquiteturais

### 1. ELT (Extract-Load-Transform)

O pipeline segue o padrão ELT moderno em vez do tradicional ETL:

- **Extract**: MongoDB → Landing (JSON bruto)
- **Load**: Landing → Bronze (Delta Lake, schema-on-read)
- **Transform**: Bronze → Silver → Gold (processamento em camadas, com qualidade progressiva)

**Vantagem**: Os dados são persistidos em cada camada, permitindo reprocessamento, debug e análise de qualidade sem re-extrair da origem.

### 2. Lambda Architecture (Lightweight)

Embora não seja uma implementação completa de Lambda Architecture, o sistema adota o conceito de camadas de dados brutos e refinados:

- **Camada Bruta (Landing/Bronze)**: Dados em formato original, alta volume, baixa latência de ingestão
- **Camada Refinada (Silver/Gold)**: Dados limpos, modelados, com alta latência de processamento mas alta qualidade

### 3. Slowly Changing Dimensions (SCD) Tipo 2

Aplicado nas dimensões da camada Gold para preservar histórico de alterações de atributos:

| Aspecto | Implementação |
|---------|--------------|
| **Chave natural** | `cliente_key` (ID original do cliente) |
| **Surrogate key** | `cliente_sk` (SHA256 do ID + data de vigência) |
| **Vigência** | `dw_valid_from` e `dw_valid_to` |
| **Flag atual** | `dw_is_current` (boolean) |
| **Hash de atributos** | `dw_record_hash` (SHA256 dos campos de negócio) |
| **Detecção de mudança** | Comparação de `dw_record_hash` entre versões |
| **Merge Delta** | `whenMatchedUpdate` + `whenNotMatchedInsert` + `whenNotMatchedBySourceDelete` |

**Vantagem**: Permite análise temporal consistente — responder "como o cliente era no momento da compra" em vez de "como o cliente é agora".

> **Referência**: `spark_jobs/silver_to_gold.py` (funções `_sync_dimension_scd2`, `_attach_dimension_sk`), `tests/test_scd2_gold_spark.py`

### 4. Checkpoint Pattern (Incremental Load)

Cada execução da DAG `mongodb_to_landing` registra o maior `updated_at` processado em uma Airflow Variable:

```text
Variable: mongodb_landing_checkpoint__ecommerce_clientes
Valor: 2026-06-13T12:30:00Z
```

A próxima execução usa este valor como filtro: `updated_at >= checkpoint - overlap_hours`.

**Vantagem**: Processa apenas dados novos ou alterados, reduzindo carga no MongoDB e volume de transferência.

> **Referência**: `dags/lib/mongodb_landing.py` (funções `checkpoint_variable_name`, `build_incremental_filter`)

### 5. Audit Trail / Data Lineage

Cada execução de job gera um **manifesto JSON** com metadados completos da execução:

| Campo | Exemplo |
|-------|---------|
| `dag_id` | `mongodb_to_landing` |
| `run_id` | `manual__2026-06-13T10:00:00+00:00` |
| `logical_date` | `2026-06-13T10:00:00Z` |
| `total_documents` | `15000` |
| `collections` | Lista com status de cada coleção |

Manifestos são gravados em `landing/_control/`, `bronze/_control/`, `silver/_control/`, `gold/_control/` com particionamento por data.

**Vantagem**: Permite auditoria completa, debugging de execuções, e análise de performance ao longo do tempo.

### 6. Data Quality Pattern (Reject + Log)

Na camada Silver, registros que não passam na validação são rejeitados e gravados em uma tabela Delta append-only:

| Campo | Significado |
|-------|-------------|
| `run_id` | ID da execução Airflow |
| `dag_id` | `bronze_to_silver` |
| `tabela` | Nome da tabela Silver |
| `chave_primaria` | Valor da PK do registro rejeitado |
| `tipo_rejeicao` | `cpf_duplicado`, `pagamento_duplicado_aprovado` |
| `detalhe` | Descrição legível da regra violada |
| `hash_registro` | SHA256 dos campos de negócio |
| `rejeitado_em` | Timestamp UTC |

**Vantagem**: Separação clara entre dados válidos (Silver) e inválidos (quality_log), permitindo análise de qualidade sem poluir a camada analítica.

> **Referência**: `spark_jobs/bronze_to_silver.py` (função `_write_quality_log`)

### 7. Idempotência

Todas as operações são idempotentes — podem ser executadas múltiplas vezes sem efeitos colaterais:

| Componente | Mecanismo de Idempotência |
|-----------|---------------------------|
| `carregar_mongo.py` | Recria coleções (`drop` + `create`) a cada execução |
| `criar_estrutura_*.py` | Verifica existência de objetos antes de criar |
| Landing → Bronze | `left_anti` join em `_bronze_source_file` evita reprocessar arquivos já ingeridos |
| Bronze → Silver | MERGE compara `updated_at` + `_silver_record_hash` para evitar duplicar |
| Silver → Gold | MERGE compara `_gold_record_hash` para sincronizar sem duplicar |
| SCD Tipo 2 | Hash de atributos detecta alterações; versões inalteradas não geram escrita |

### 8. Separation of Concerns (SoC)

A lógica de negócio está separada em módulos puros (`dags/lib/`), independentes do Airflow:

| Módulo | Responsabilidade | Independente de Airflow? |
|--------|-----------------|-------------------------|
| `dags/lib/mongodb_landing.py` | Checkpoints, filtros, manifestos | Sim |
| `dags/lib/landing_bronze.py` | URIs, Spark config, manifestos | Sim |
| `dags/lib/bronze_silver.py` | Regras de entidade, qualidade, MERGE | Sim |
| `dags/lib/silver_gold.py` | Modelos Gold, SCD2, manifestos | Sim |

As DAGs (`dags/*.py`) contêm apenas orquestração (tasks, agendamento, retries), permitindo testes unitários dos helpers sem inicializar o Airflow.

---

## Fluxos Síncronos e Assíncronos

### Fluxo Síncrono (Pipeline Principal)

O fluxo principal é **síncrono e sequencial** dentro de cada DAG, mas **assíncrono entre DAGs** (agendamento escalonado):

```mermaid
sequenceDiagram
    participant MongoDB as MongoDB Atlas
    participant Airflow as Apache Airflow
    participant MinIO as MinIO Data Lake
    participant Spark as Spark Cluster

    Note over MongoDB,MinIO: DAG 1: mongodb_to_landing (min 0)
    Airflow->>MongoDB: find(updated_at >= checkpoint)
    MongoDB-->>Airflow: Cursor de documentos (batch 1000)
    Airflow->>Airflow: Serializa para JSON Lines
    Airflow->>MinIO: PUT landing/ecommerce/.../part-00000.json
    Airflow->>Airflow: Atualiza checkpoint Variable
    Airflow->>MinIO: PUT manifest.json

    Note over MongoDB,MinIO: DAG 2: landing_to_bronze (min 5)
    Airflow->>MinIO: LIST landing/ecommerce/.../*.json
    MinIO-->>Airflow: Lista de arquivos JSON
    Airflow->>Spark: spark-submit landing_to_bronze.py
    Spark->>MinIO: READ JSON Lines (S3A)
    Spark->>Spark: Adiciona metadados de auditoria
    Spark->>MinIO: WRITE Delta Lake (bronze/ecommerce/.../)
    Spark->>MinIO: PUT manifest.json
    Spark-->>Airflow: Retorna status

    Note over MongoDB,MinIO: DAG 3: bronze_to_silver (min 10)
    Airflow->>Spark: spark-submit bronze_to_silver.py
    Spark->>MinIO: READ Delta Bronze (S3A)
    Spark->>Spark: Deduplica, valida, aplica regras de negócio
    Spark->>MinIO: WRITE Delta Silver (MERGE incremental)
    Spark->>MinIO: WRITE quality_log (rejeições)
    Spark->>MinIO: PUT manifest.json
    Spark-->>Airflow: Retorna status

    Note over MongoDB,MinIO: DAG 4: silver_to_gold (min 15)
    Airflow->>Spark: spark-submit silver_to_gold.py
    Spark->>MinIO: READ Delta Silver (S3A)
    Spark->>Spark: Build dimensões (SCD2) + Fatos
    Spark->>MinIO: WRITE Delta Gold (MERGE + SCD2)
    Spark->>MinIO: PUT manifest.json
    Spark-->>Airflow: Retorna status
```

### Fluxo Assíncrono (Agendamento)

As 4 DAGs são agendadas com **offset** dentro de um ciclo de 15 minutos, simulando near real-time:

| DAG | Schedule | Offset |
|-----|----------|--------|
| `mongodb_to_landing` | `*/15 * * * *` | 0 min |
| `landing_to_bronze` | `5-59/15 * * * *` | 5 min |
| `bronze_to_silver` | `10-59/15 * * * *` | 10 min |
| `silver_to_gold` | `15-59/15 * * * *` | 15 min |

Isso permite que cada DAG processe o resultado da anterior dentro do mesmo ciclo, sem necessidade de dependências explícitas (embora na prática a DAG 2 dependa da 1, etc.).

---

## Preocupações Transversais (Cross-cutting Concerns)

### Auditoria e Data Lineage

Cada registro em cada camada carrega metadados de auditoria:

| Camada | Colunas de Auditoria |
|--------|---------------------|
| Landing | `extraction_date`, `run_id` (no path) |
| Bronze | `_bronze_source_file`, `_bronze_extraction_date`, `_bronze_landing_run_id`, `_bronze_airflow_run_id`, `_bronze_ingested_at`, `ingestion_date` |
| Silver | `_silver_source_file`, `_silver_source_ingested_at`, `_silver_record_hash`, `_silver_airflow_run_id`, `_silver_processed_at` |
| Gold | `_gold_record_hash`, `_gold_airflow_run_id`, `_gold_processed_at` |
| SCD2 | `dw_valid_from`, `dw_valid_to`, `dw_is_current`, `dw_record_hash` |

### Logging

- Jobs PySpark usam `logging.getLogger(__name__)` com nível INFO/WARN
- Spark UI log level configurado via `SPARK_LOG_LEVEL` (default: WARN)
- Airflow logs persistem em `airflow/logs/` (volume Docker, não versionado)

### Retry e Resiliência

| DAG | Retries | Retry Delay | Timeout |
|-----|---------|-------------|---------|
| `mongodb_to_landing` | 2 | 2 min | — |
| `landing_to_bronze` | 2 | 5 min | 2 horas |
| `bronze_to_silver` | 2 | 5 min | 2 horas |
| `silver_to_gold` | 2 | 5 min | 2 horas |

### Segurança (Configurável)

- Credenciais em `.env` (não versionado)
- Airflow Connections (`mongodb_atlas`, `minio_s3`) configuradas via variáveis de ambiente
- JWT secret para API do Airflow configurável via `AIRFLOW__API_AUTH__JWT_SECRET`
